Students refer to pseudo codes of BFS, DFS, UCS, DLS, and IDS in [this link](https://drive.google.com/file/d/1q2LtrRCfemfiqyhfxNMcVJ3alvLh_pdV/view?usp=share_link) to implement the corresponding classes in TODO 1 - 5. \
Students can add supporting attributes and methods to the five classes of search strategies as needed.

# Libraries

In [1]:
import os
import heapq

# Graph class

In [2]:
# Directed, weighted graphs
class Graph:
  def __init__(self):
    self.AL = dict() # adjacency list
    self.V = 0
    self.E = 0

  def __str__(self):
    res = 'V: %d, E: %d\n'%(self.V, self.E)
    for u, neighbors in self.AL.items():
      line = '%d: %s\n'%(u, str(neighbors))
      res += line
    return res

  def print(self):
    print(str(self))

  def load_from_file(self, filename):
    '''
        Example input file:
            V E
            u v w
            u v w
            u v w
            ...

        # input.txt
        7 8
        0 1 5
        0 2 6
        1 3 12
        1 4 9
        2 5 5
        3 5 8
        3 6 7
        4 6 4
    '''
    if os.path.exists(filename):
      with open(filename) as g:
        self.V, self.E = [int(it) for it in g.readline().split()]
        for line in g:
          u, v, w = [int(it) for it in line.strip().split()]
          if u not in self.AL:
            self.AL[u] = []
          self.AL[u].append((v, w))

In [3]:
g = Graph()
g.load_from_file('input.txt')
g.print()

V: 7, E: 8
0: [(1, 5), (2, 6)]
1: [(3, 12), (4, 9)]
2: [(5, 5)]
3: [(5, 8), (6, 7)]
4: [(6, 4)]



# Search Strategies

In [4]:
class SearchStrategy:
  def search(self, g: Graph, src: int, dst: int) -> tuple:
    expanded = [] # list of expanded vertices in the traversal order
    path = [] # path from src to dst
    return expanded, path

In [8]:
class BFS(SearchStrategy):
    def search(self, g: Graph, src: int, dst: int) -> tuple:
        expanded = [] # list of expanded vertices in the traversal order
        path = [] # path from src to dst

        # --- TODO 1 Starts Here ---

        # 1. Handle Edge Case: Source is already the Destination
        if src == dst:
            return [src], [src]

        # 2. Initialize Data Structures
        # frontier: A FIFO queue to store nodes waiting to be explored
        # explored: A set to keep track of nodes we have already discovered
        # parent: A dictionary to store the "footprints" (who discovered whom)
        frontier = [src]
        explored = {src}
        parent = {src: None}

        # 3. Core Search Loop
        while frontier:
            # Pop the first element from the list (First-In, First-Out)
            current_node = frontier.pop(0)
            expanded.append(current_node)

            # Check if we have reached the destination
            if current_node == dst:
                break

            # 4. Explore Neighbors
            if current_node in g.AL:
                # Iterate through all adjacent nodes (neighbors)
                for neighbor, _ in g.AL[current_node]:
                    # Only process neighbors we haven't seen before
                    if neighbor not in explored:
                        explored.add(neighbor)          # Mark as discovered immediately
                        parent[neighbor] = current_node # Record the parent for path reconstruction
                        frontier.append(neighbor)       # Add to the end of the queue

        # 5. Path Reconstruction (Backtracking from Destination to Source)
        if dst in parent:
            step = dst
            while step is not None:
                path.append(step)
                step = parent[step] # Move one step back to the parent

            # Reverse the path to get the correct order from Source to Destination
            path.reverse()

        return expanded, path

In [9]:
class DFS(SearchStrategy):
    def search(self, g: Graph, src: int, dst: int) -> tuple:
        expanded = [] # list of expanded vertices in the traversal order
        path = [] # path from src to dst

        # --- TODO 2 Starts Here ---

        # 1. Handle Edge Case: Source is already the Destination
        if src == dst:
            return [src], [src]

        # 2. Initialize Data Structures
        # frontier: A LIFO stack (Last-In, First-Out)
        # explored: A set to keep track of nodes already visited
        # parent: To store the path relationship
        frontier = [src]
        explored = set()
        parent = {src: None}

        # 3. Core Search Loop
        while frontier:
            # Pop the LAST element added (LIFO behavior)
            current_node = frontier.pop()

            # In DFS, a node might be added to the stack multiple times
            # from different parents. We skip it if we already processed it.
            if current_node in explored:
                continue

            # Mark node as visited and add to expansion order
            expanded.append(current_node)
            explored.add(current_node)

            # Check if we reached the goal
            if current_node == dst:
                break

            # 4. Explore Neighbors
            if current_node in g.AL:
                # We reverse the neighbors so that the first neighbor in the list
                # ends up at the top of the stack and gets processed first.
                for neighbor, _ in reversed(g.AL[current_node]):
                    if neighbor not in explored:
                        # Update/Set the parent for path reconstruction
                        parent[neighbor] = current_node
                        frontier.append(neighbor)

        # 5. Path Reconstruction (Backtracking)
        if dst in parent:
            step = dst
            while step is not None:
                path.append(step)
                step = parent[step]

            path.reverse()

        return expanded, path

In [10]:
class UCS(SearchStrategy):
    def search(self, g: Graph, src: int, dst: int) -> tuple:
        expanded = [] # list of expanded vertices in the traversal order
        path = [] # path from src to dst

        # --- TODO 3 Starts Here ---
        # Priority Queue stores tuples: (cumulative_cost, current_node)
        frontier = [(0, src)]
        # Dictionary to keep track of the minimum cost to reach each node
        min_cost = {src: 0}
        parent = {src: None}

        while frontier:
            # Pop the node with the lowest cumulative cost
            cost, current_node = heapq.heappop(frontier)

            # Standard UCS: Only add to expanded when the node is "popped"
            if current_node in expanded:
                continue
            expanded.append(current_node)

            # Goal test at expansion time
            if current_node == dst:
                break

            # Explore neighbors
            if current_node in g.AL:
                for neighbor, weight in g.AL[current_node]:
                    new_cost = cost + weight

                    # If neighbor is new or we found a cheaper path to it
                    if neighbor not in min_cost or new_cost < min_cost[neighbor]:
                        min_cost[neighbor] = new_cost
                        parent[neighbor] = current_node
                        heapq.heappush(frontier, (new_cost, neighbor))

        # Path reconstruction
        if dst in parent:
            step = dst
            while step is not None:
                path.append(step)
                step = parent[step]
            path.reverse()

        return expanded, path

In [11]:
class DLS(SearchStrategy):
    def __init__(self, LIM: int):
        self.LIM = LIM

    def recursive_dls(self, g: Graph, node: int, dst: int, limit: int,
                      expanded: list, parent: dict) -> bool:
        # Add node to expansion order
        expanded.append(node)

        # Base case: Goal reached
        if node == dst:
            return True

        # Base case: Depth limit reached
        if limit <= 0:
            return False

        # Recursive step: Expand neighbors
        if node in g.AL:
            for neighbor, _ in g.AL[node]:
                # In DLS, we avoid cycles by checking if the neighbor is not in current path
                # However, for simplicity here, we check against expanded or parents
                parent[neighbor] = node
                if self.recursive_dls(g, neighbor, dst, limit - 1, expanded, parent):
                    return True
        return False

    def search(self, g: Graph, src: int, dst: int) -> tuple:
        expanded = []
        path = []
        parent = {src: None}

        # Start recursion from depth 0 up to self.LIM
        if self.recursive_dls(g, src, dst, self.LIM, expanded, parent):
            # Reconstruct path if goal found
            step = dst
            while step is not None:
                path.append(step)
                step = parent.get(step)
            path.reverse()

        return expanded, path

In [12]:
class IDS(SearchStrategy):
    def __init__(self, MAX_LIM: int):
        self.MAX_LIM = MAX_LIM

    def search(self, g: Graph, src: int, dst: int) -> tuple:
        all_expanded = [] # To track expansion across all iterations

        # Incrementally increase the depth limit
        for depth in range(self.MAX_LIM + 1):
            dls = DLS(depth)
            expanded_in_this_run, path = dls.search(g, src, dst)

            # Accumulate expanded nodes for the final report
            all_expanded.extend(expanded_in_this_run)

            # If path is found, return immediately
            if path:
                return all_expanded, path

        # If no path found within MAX_LIM
        return all_expanded, []

# Evaluation

In [13]:
bfs = BFS()
dfs = DFS()
ucs = UCS()
dls = DLS(LIM=3)
ids = IDS(MAX_LIM=5)

for stg in [bfs, dfs, ucs, dls, ids]:
  print(stg)
  expanded, path = stg.search(g, 0, g.V-1)
  print(expanded)
  print(path)




[0, 1, 2, 3, 4, 5, 6]
[0, 1, 3, 6]
[0, 1, 3, 5, 6]
[0, 1, 3, 6]
[0, 1, 2, 5, 4, 3, 6]
[0, 1, 4, 6]
[0, 1, 3, 5, 6]
[0, 1, 3, 6]
[0, 0, 1, 2, 0, 1, 3, 4, 2, 5, 0, 1, 3, 5, 6]
[0, 1, 3, 6]


# Submission

*   Students download the notebook after completion
*   Rename the notebook in which inserting your student ID at the beginning. \
For example, **123456-lec04-UninformedSearch-HW.ipynb**
*   Finally, submit the file

